In [4]:
from nuscenes.nuscenes import NuScenes

nusc = NuScenes(version='v1.0-trainval', dataroot='/cpfs01/user/yangjiazhi/workspace/DVGen/CogVideo/nuscenes', verbose=True)
dataset = nusc

Loading NuScenes tables for version v1.0-trainval...
23 category,
8 attribute,
4 visibility,
64386 instance,
12 sensor,
10200 calibrated_sensor,
2631083 ego_pose,
68 log,
850 scene,
34149 sample,
2631083 sample_data,
1166187 sample_annotation,
4 map,
Done loading in 28.677 seconds.
Reverse indexing ...
Done reverse indexing in 6.8 seconds.


In [6]:
import numpy as np
from tqdm import tqdm

In [15]:
def get_sample_indices(samples, num_frames, interval):
    indices = list()
    total_capacity = len(samples)
    num_all_frames = num_frames * interval

    for index in tqdm(range(total_capacity), desc="Indexing frame clips"):
        is_valid_data = True
        previous_sample = None
        current_indices = list()
        for t in range(num_all_frames):
            index_t = index + t
            # exceed the dataset capacity
            if index_t >= total_capacity:
                is_valid_data = False
                break
            current_sample = samples[index_t]
            # check if scene is the same
            if previous_sample is not None and current_sample["scene_token"] != previous_sample["scene_token"]:
                is_valid_data = False
                break

            if t % interval == 0:
                current_indices.append(index_t)
            previous_sample = current_sample

        if is_valid_data:
            indices.append(current_indices)
    return np.asarray(indices)

In [8]:
dataset = nusc

In [55]:
# samples = list()
# for sample in dataset.sample:
#     if sample["scene_token"] in scenes:
#         self.samples.append(sample)
samples = dataset.sample
samples.sort(key=lambda x: (x["scene_token"], x["timestamp"]))

In [19]:
num_frames = 25
interval = 1
sample_indices = get_sample_indices(samples, num_frames, interval)

Indexing frame clips: 100%|██████████| 34149/34149 [00:00<00:00, 261078.92it/s]


In [22]:
sample_indices

array([[    0,     1,     2, ...,    22,    23,    24],
       [    1,     2,     3, ...,    23,    24,    25],
       [    2,     3,     4, ...,    24,    25,    26],
       ...,
       [34122, 34123, 34124, ..., 34144, 34145, 34146],
       [34123, 34124, 34125, ..., 34145, 34146, 34147],
       [34124, 34125, 34126, ..., 34146, 34147, 34148]])

In [27]:
first_sample = samples[sample_indices[0][0]]
first_sample.keys()

dict_keys(['token', 'timestamp', 'prev', 'next', 'scene_token', 'data', 'anns'])

In [37]:
dataset.scene[0]['first_sample_token']

'e93e98b63d3b40209056d129dc53ceee'

In [51]:
dataset.get('sample_data', first_sample['data']['LIDAR_TOP'])

{'token': '5721ccb590c84115b1d229306cf77363',
 'sample_token': '3481dbfd65864925b4a4b2d6b7256d44',
 'ego_pose_token': '5721ccb590c84115b1d229306cf77363',
 'calibrated_sensor_token': '458cebce10b34f998a4c3f80988bb8f4',
 'timestamp': 1532708429048702,
 'fileformat': 'pcd',
 'is_key_frame': True,
 'height': 0,
 'width': 0,
 'filename': 'samples/LIDAR_TOP/n008-2018-07-27-12-07-38-0400__LIDAR_TOP__1532708429048702.pcd.bin',
 'prev': '',
 'next': 'd94d331c933242d68f44b877fcaf434a',
 'sensor_modality': 'lidar',
 'channel': 'LIDAR_TOP'}

In [52]:
dataset.get('sample_data', first_sample['data']['LIDAR_TOP'])['next']

'd94d331c933242d68f44b877fcaf434a'

In [53]:
dataset.get('sample_data', 'd94d331c933242d68f44b877fcaf434a')

{'token': 'd94d331c933242d68f44b877fcaf434a',
 'sample_token': '394d87634b6c46049c2f06e84026096a',
 'ego_pose_token': 'd94d331c933242d68f44b877fcaf434a',
 'calibrated_sensor_token': '458cebce10b34f998a4c3f80988bb8f4',
 'timestamp': 1532708429098481,
 'fileformat': 'pcd',
 'is_key_frame': False,
 'height': 0,
 'width': 0,
 'filename': 'sweeps/LIDAR_TOP/n008-2018-07-27-12-07-38-0400__LIDAR_TOP__1532708429098481.pcd.bin',
 'prev': '5721ccb590c84115b1d229306cf77363',
 'next': '7e262fa8320c4415b28f538c194329a4',
 'sensor_modality': 'lidar',
 'channel': 'LIDAR_TOP'}

In [54]:
dataset.get('sample_data', '7e262fa8320c4415b28f538c194329a4')

{'token': '7e262fa8320c4415b28f538c194329a4',
 'sample_token': '394d87634b6c46049c2f06e84026096a',
 'ego_pose_token': '7e262fa8320c4415b28f538c194329a4',
 'calibrated_sensor_token': '458cebce10b34f998a4c3f80988bb8f4',
 'timestamp': 1532708429148798,
 'fileformat': 'pcd',
 'is_key_frame': False,
 'height': 0,
 'width': 0,
 'filename': 'sweeps/LIDAR_TOP/n008-2018-07-27-12-07-38-0400__LIDAR_TOP__1532708429148798.pcd.bin',
 'prev': 'd94d331c933242d68f44b877fcaf434a',
 'next': '6c9cb39957d04338b09b8c809150baea',
 'sensor_modality': 'lidar',
 'channel': 'LIDAR_TOP'}

In [5]:
# videos = []  # * 12 hz

sweeps = []
samples = dataset.sample
samples.sort(key=lambda x: (x["scene_token"], x["timestamp"]))

sensor = 'CAM_FRONT'
num_frames = 49
# video_clip = []
scene_token = samples[0]["scene_token"]

def get_sensor_from_sensor_token(sensor_token):
    return dataset.get('sample_data', sensor_token)

def get_sample_token_from_sensor_token(sensor_token):
    return dataset.get('sample_data', sensor_token)['sample_token']

def get_scene_token_from_sensor_token(sensor_token):
    # sample_token = dataset.get('sample_data', sensor_token)['sample_token']
    sample_token = get_sample_token_from_sensor_token(sensor_token)
    return dataset.get('sample', sample_token)['scene_token']


# * Use prev, not next
for sample_ind, sample in enumerate(samples):
    
    sweep_clip = []
    sample_token = sample['token']
    
    cur_cam_token = dataset.get('sample_data', sample['data'][sensor])['token']
    cur_sample_token = get_sample_token_from_sensor_token(cur_cam_token)

    indi = 0
    while cur_sample_token == sample_token:
        cur_sensor = get_sensor_from_sensor_token(cur_cam_token)
        indi += 1
        # sweep_clip.append(
        #     cur_sensor['filename']
        # )
        sweep_clip.insert(
            0,
            cur_sensor['filename']
        )

        cur_cam_token = cur_sensor['prev']
        if cur_cam_token == '':
            break
        cur_sample_token = get_sample_token_from_sensor_token(cur_cam_token)
    
    sweeps.append(sweep_clip)

In [23]:
len(sweeps)

34149

In [26]:
set([len(s) for s in sweeps])

{1, 3, 4, 5, 6, 7, 8}

In [30]:
len([s for s in sweeps if len(s) == 1])

850

In [22]:
# * Visualize
import os
img_root = '/cpfs01/user/yangjiazhi/workspace/DVGen/CogVideo/nuscenes'
img_list = [os.path.join(img_root, img) for img in sweeps[3]]
print(img_list)
# def img_list_to_video(img_list, out_path):
#     import cv2
#     img = cv2.imread(img_list[0])
#     height, width, layers = img.shape

#     video = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), 1, (width, height))

#     for img in img_list:
#         video.write(cv2.imread(img))

#     cv2.destroyAllWindows()
#     video.release()


# * imageio is better in quality
def img_list_to_video_with_imageio(img_list, out_path, fps=12):
    import imageio
    writer = imageio.get_writer(out_path, fps=fps, macro_block_size=1)  # * macro_block_size=1, compatibility
    for img in img_list:
        writer.append_data(imageio.imread(img))
    writer.close()

# img_list_to_video(img_list, 'test.mp4')
img_list_to_video_with_imageio(img_list, 'test_imageio.mp4')

['/cpfs01/user/yangjiazhi/workspace/DVGen/CogVideo/nuscenes/sweeps/CAM_FRONT/n008-2018-07-27-12-07-38-0400__CAM_FRONT__1532708430112404.jpg', '/cpfs01/user/yangjiazhi/workspace/DVGen/CogVideo/nuscenes/sweeps/CAM_FRONT/n008-2018-07-27-12-07-38-0400__CAM_FRONT__1532708430262404.jpg', '/cpfs01/user/yangjiazhi/workspace/DVGen/CogVideo/nuscenes/sweeps/CAM_FRONT/n008-2018-07-27-12-07-38-0400__CAM_FRONT__1532708430364520.jpg', '/cpfs01/user/yangjiazhi/workspace/DVGen/CogVideo/nuscenes/sweeps/CAM_FRONT/n008-2018-07-27-12-07-38-0400__CAM_FRONT__1532708430412404.jpg', '/cpfs01/user/yangjiazhi/workspace/DVGen/CogVideo/nuscenes/samples/CAM_FRONT/n008-2018-07-27-12-07-38-0400__CAM_FRONT__1532708430512404.jpg']


/tmp/ipykernel_319860/3878272121.py:24: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  writer.append_data(imageio.imread(img))
[rawvideo @ 0x647b940] Stream #0: not enough frames to estimate rate; consider increasing probesize


In [35]:
my_scene = dataset.scene[0]
my_scene
# my_sample = dataset.sample[0]
# my_sample.keys()

{'token': '73030fb67d3c46cfb5e590168088ae39',
 'log_token': '6b6513e6c8384cec88775cae30b78c0e',
 'nbr_samples': 40,
 'first_sample_token': 'e93e98b63d3b40209056d129dc53ceee',
 'last_sample_token': '40e413c922184255a94f08d3c10037e0',
 'name': 'scene-0001',
 'description': 'Construction, maneuver between several trucks'}

In [36]:
# videos = []  # * 12 hz

# sweeps = []
sweeps = dict()  # scene_token -> sweep_clip
samples = dataset.sample
samples.sort(key=lambda x: (x["scene_token"], x["timestamp"]))

sensor = 'CAM_FRONT'
num_frames = 49
# video_clip = []
scene_token = samples[0]["scene_token"]

def get_sensor_from_sensor_token(sensor_token):
    return dataset.get('sample_data', sensor_token)

def get_sample_token_from_sensor_token(sensor_token):
    return dataset.get('sample_data', sensor_token)['sample_token']

def get_scene_token_from_sensor_token(sensor_token):
    # sample_token = dataset.get('sample_data', sensor_token)['sample_token']
    sample_token = get_sample_token_from_sensor_token(sensor_token)
    return dataset.get('sample', sample_token)['scene_token']


# * Use prev, not next
for sample_ind, sample in enumerate(samples):
    
    sweep_clip = []
    sample_token = sample['token']
    scene_token = sample['scene_token']

    if scene_token not in sweeps:
        sweeps[scene_token] = dict()
        sweeps[scene_token]['sweep'] = []
        sweeps[scene_token]['desc'] = dataset.get('scene', scene_token)['description']
    
    cur_cam_token = dataset.get('sample_data', sample['data'][sensor])['token']
    cur_sample_token = get_sample_token_from_sensor_token(cur_cam_token)

    indi = 0
    while cur_sample_token == sample_token:
        cur_sensor = get_sensor_from_sensor_token(cur_cam_token)
        indi += 1
        # sweep_clip.append(
        #     cur_sensor['filename']
        # )
        sweep_clip.insert(
            0,
            cur_sensor['filename']
        )

        cur_cam_token = cur_sensor['prev']
        if cur_cam_token == '':
            break
        cur_sample_token = get_sample_token_from_sensor_token(cur_cam_token)
    
    # sweeps.append(sweep_clip)
    sweeps[scene_token]['sweep'].extend(sweep_clip)

In [39]:
len(sweeps.keys())  # * 850 scenes

850

In [43]:

from tqdm import tqdm

def img_list_to_video_with_imageio(img_list, out_path, fps=12):
    import imageio
    writer = imageio.get_writer(out_path, fps=fps, macro_block_size=1)  # * macro_block_size=1, compatibility
    for img in img_list:
        writer.append_data(imageio.imread(img))
    writer.close()

img_root = '/cpfs01/user/yangjiazhi/workspace/DVGen/CogVideo/nuscenes'
label_root = '/cpfs01/user/yangjiazhi/workspace/DVGen/CogVideo/debug/nus_sweeps/labels'
videos_root = '/cpfs01/user/yangjiazhi/workspace/DVGen/CogVideo/debug/nus_sweeps/videos'
for scene_token in tqdm(sweeps.keys()):
    # save description to txt  # in labels/{scene_token}.txt
    # save sweep_clip to mp4  # in videos/{scene_token}.mp4
    desc = sweeps[scene_token]['desc']
    sweep_clip = sweeps[scene_token]['sweep']
    sweep_clip = [os.path.join(img_root, img) for img in sweep_clip]

    with open(os.path.join(label_root, f"{scene_token}.txt"), 'w') as f:
        f.write(desc)

    img_list_to_video_with_imageio(sweep_clip, os.path.join(videos_root, f"{scene_token}.mp4"), fps=12)

/tmp/ipykernel_319860/23134592.py:5: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  writer.append_data(imageio.imread(img))


KeyboardInterrupt: 